# Project 08 — Many Predictors, Sparse Truth (Regularized Horseshoe)

**Scenario.** We have ~20 candidate predictors (assay readouts, expression markers, engineered covariates) and want to know which *few* actually drive a continuous response. The truth is **sparse**: only 3 of the 20 coefficients are nonzero; the rest are exactly zero.

**New skill.** *Shrinkage priors and LOO/WAIC comparison.* A wide-Normal ("ridge") prior spreads small spurious effects across all predictors. The **horseshoe** prior shrinks the noise coefficients hard toward zero while letting the few real ones escape — recovering sparsity. We compare the two with LOO and discuss the **double-dipping / selection-bias** trap.

In [ ]:
import sys, pathlib
sys.path.insert(0, r'/home/user/biofx_python/bayesian_workflow_portfolio')
sys.path.insert(0, str(pathlib.Path.cwd()))
import warnings; warnings.filterwarnings('ignore')

In [ ]:
import numpy as np
import pymc as pm
import arviz as az
import matplotlib.pyplot as plt
az.style.use('arviz-darkgrid')
RNG = 20240601

## Step 1 — Problem & data-generating story

$$y_i=\beta_0+\sum_{j=1}^{P} X_{ij}\beta_j+\varepsilon_i,\quad \varepsilon_i\sim N(0,\sigma),\quad \text{only } K\ll P \text{ of the }\beta_j\neq 0.$$

**Assumptions made explicit:** (a) the response is linear in the predictors; (b) the true coefficient vector is *sparse*; (c) predictor columns are standardized so coefficient magnitudes are comparable; (d) errors are Normal and homoscedastic. Truth: nonzero at indices 2, 7, 13 with values 2.5, -1.8, 1.4; all other 17 coefficients are exactly 0.

In [ ]:
from data.generate_data import generate
data = generate()
X, y = data['X'], data['y']
print(f"n={data['n']}, p={data['p']}, truly nonzero at {data['nonzero_idx']}")
print('truth:', {f'beta[{j}]': round(float(data['beta_true'][j]),2)
                 for j in data['nonzero_idx']})

## Step 2 — Two priors: wide-Normal ridge vs regularized horseshoe

Both share $y=\beta_0+X\beta+N(0,\sigma)$; only the prior on $\beta$ differs.

**Ridge:** $\beta_j\sim N(0,5)$ — every coefficient is equally free; no sparsity. **Horseshoe:** each $\beta_j$ has scale $\tau\,\tilde\lambda_j$, where the *global* $\tau$ controls how many coefficients survive and the *local* $\lambda_j$ has heavy (half-Cauchy) tails so a coefficient is either crushed to ~0 or allowed to escape. The **regularized** variant caps the escaping magnitude via a slab, which stabilizes sampling.

**Crucial implementation detail — non-centered parameterization.** We sample standardized $z_j\sim N(0,1)$ and set $\beta_j=z_j\,\tau\,\tilde\lambda_j$. The *centered* version (sampling $\beta_j\sim N(0,\tau\lambda_j)$ directly) creates a pinched funnel and floods the run with divergences — that is the seeded bug in the broken notebook.

In [ ]:
from model import build_model, fit
m_ridge = build_model(data, model='ridge')
m_hs = build_model(data, model='horseshoe')
m_hs

## Step 3 — Prior predictive check

We confirm the horseshoe prior implies a sparse-ish coefficient vector: most prior-drawn coefficients sit near zero with a few heavy-tailed escapes — exactly the structure we believe in.

In [ ]:
with m_hs:
    prior = pm.sample_prior_predictive(draws=400, random_seed=RNG)
pb = prior.prior['beta'].values.reshape(-1)
pb = pb[np.abs(pb) < np.percentile(np.abs(pb), 98)]
fig, ax = plt.subplots(figsize=(6, 3.4))
ax.hist(pb, bins=60, color='#4C72B0', edgecolor='white')
ax.set(xlabel='prior-drawn coefficient (98th-pct clipped)', ylabel='count',
       title='Horseshoe prior — spike near 0 with heavy tails')
plt.tight_layout()

## Step 4 — Inference (NUTS) for both models

Settings: `draws=1000, tune=1000, chains=4, target_accept=0.95`. The high `target_accept` shrinks the step size, which the horseshoe geometry needs even when non-centered. We keep both idatas.

In [ ]:
idata_ridge = fit(data, model='ridge', draws=1000, tune=1000, chains=4, seed=101)
idata_hs = fit(data, model='horseshoe', draws=1000, tune=1000, chains=4, seed=101)
print('ridge divergences    :', int(idata_ridge.sample_stats['diverging'].sum()))
print('horseshoe divergences:', int(idata_hs.sample_stats['diverging'].sum()))

## Step 5 — Computational diagnostics

For the non-centered horseshoe, divergences should be few or zero and $\hat R\approx1$. **If divergences flood in, suspect a centered parameterization** (the classic horseshoe funnel) — that is the diagnostic the broken notebook is built around. `az.plot_energy` and the divergence count are the tools.

In [ ]:
print(az.summary(idata_hs, var_names=['tau', 'sigma']))
az.plot_energy(idata_hs); plt.tight_layout()

## Step 6 — Posterior predictive & sparsity recovery

The headline plot: a forest of the 20 coefficients under each prior. The ridge leaves the noise coefficients scattered with non-trivial intervals; the horseshoe collapses them onto zero and cleanly isolates the 3 real signals.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
bh = idata_hs.posterior['beta']
br = idata_ridge.posterior['beta']
idx = np.arange(data['p'])
mh = bh.mean(('chain','draw')).values
hh = az.hdi(bh, hdi_prob=0.94)['beta'].values
mr = br.mean(('chain','draw')).values
hr = az.hdi(br, hdi_prob=0.94)['beta'].values
ax.errorbar(mr, idx+0.15, xerr=[mr-hr[:,0], hr[:,1]-mr], fmt='o',
            color='#C44E52', label='ridge', alpha=0.7)
ax.errorbar(mh, idx-0.15, xerr=[mh-hh[:,0], hh[:,1]-mh], fmt='o',
            color='#55A868', label='horseshoe')
for j in data['nonzero_idx']:
    ax.scatter(data['beta_true'][j], j, color='k', marker='|', s=200, zorder=5)
ax.axvline(0, color='gray', ls=':')
ax.set(xlabel='coefficient', ylabel='predictor index',
       title='Horseshoe recovers sparsity; ridge does not (| = truth)')
ax.legend(); plt.tight_layout()

In [ ]:
az.plot_ppc(idata_hs, num_pp_samples=100); plt.tight_layout()

## Step 7 — Model comparison (LOO) & recovery

We compare the two priors with PSIS-LOO. The horseshoe typically matches or beats the ridge in expected predictive density while using **far fewer effective parameters** (`p_loo`) — it pays for the same fit with less complexity. We then confirm recovery of the nonzero coefficients.

In [ ]:
cmp = az.compare({'ridge': idata_ridge, 'horseshoe': idata_hs}, ic='loo')
print(cmp[['rank', 'elpd_loo', 'p_loo', 'elpd_diff', 'dse', 'weight']])

In [ ]:
mh = idata_hs.posterior['beta'].mean(('chain','draw')).values
zeros = [j for j in range(data['p']) if j not in data['nonzero_idx']]
print('recovered nonzero:', {j: round(float(mh[j]),2) for j in data['nonzero_idx']})
print('max |coef| among true-zeros (horseshoe):', round(float(np.max(np.abs(mh[zeros]))),3))
mr = idata_ridge.posterior['beta'].mean(('chain','draw')).values
print('max |coef| among true-zeros (ridge)    :', round(float(np.max(np.abs(mr[zeros]))),3))

## Step 8 — Decision, communication & the double-dipping warning

**The selection-bias trap.** A tempting but WRONG workflow: fit the model, pick the predictor with the largest coefficient, then re-fit *only* that predictor and report its (now tiny) p-value or (now narrow) interval as 'significance'. This **double-dips** — using the same data to *select* and to *test* — and grossly overstates confidence. The Bayesian remedy is to let the **shrinkage prior** do selection *within one joint fit* and report the full posterior over all coefficients, including the uncertainty about which are nonzero. We never re-fit on a selected subset.

In [ ]:
# Honest reporting: posterior probability each |coef| exceeds a small threshold.
beta = idata_hs.posterior['beta'].values.reshape(-1, data['p'])
thr = 0.2
p_active = (np.abs(beta) > thr).mean(0)
for j in range(data['p']):
    flag = '  <-- truly nonzero' if j in data['nonzero_idx'] else ''
    if p_active[j] > 0.2 or j in data['nonzero_idx']:
        print(f'beta[{j:2d}]: P(|coef|>{thr}) = {p_active[j]:.2f}{flag}')

**Conclusion (for a collaborator).** Three predictors (indices 2, 7, 13) drive the response; the other 17 are indistinguishable from zero. We report this from a single joint fit with a shrinkage prior — we did **not** cherry-pick a predictor and re-test it. See `summary_onepager.md`.